# FraudShield AI — Notebook 03: Feature Engineering

This notebook explores and creates meaningful features for fraud detection:
- Balance anomaly features
- Transaction amount patterns
- Account behavior features

**Key Rule:** Only features that would be available at real-time prediction time are used.
No post-transaction features (e.g., downstream system flags).

In [ ]:
import os, warnings, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

DATA_PATH  = '../data/raw/transactions.csv'
PROC_PATH  = '../data/processed/'
MODEL_PATH = '../models/'

df_raw = pd.read_csv(DATA_PATH)
rename_map = {
    'type': 'transaction_type', 'oldbalanceOrg': 'origin_balance_before',
    'newbalanceOrig': 'origin_balance_after', 'oldbalanceDest': 'dest_balance_before',
    'newbalanceDest': 'dest_balance_after', 'nameOrig': 'origin_account',
    'nameDest': 'destination_account', 'isFraud': 'is_fraud',
}
df_raw.rename(columns=rename_map, inplace=True)
df_raw.drop(columns=[c for c in ['isFlaggedFraud'] if c in df_raw.columns], inplace=True)
print(f'Loaded {len(df_raw):,} rows')

In [ ]:
# ── Feature 1: Balance Difference (Origin) ────────────────────────
# A legitimate transaction should reduce origin balance by ~amount
# Fraud often leaves no trail: balance doesn't change despite claimed transfer
df_raw['balance_diff_origin'] = df_raw['origin_balance_before'] - df_raw['origin_balance_after']
df_raw['balance_error_origin'] = abs(df_raw['balance_diff_origin'] - df_raw['amount'])

# ── Feature 2: Balance Difference (Destination) ───────────────────
df_raw['balance_diff_dest'] = df_raw['dest_balance_after'] - df_raw['dest_balance_before']

# ── Feature 3: Amount Relative to Origin Balance ──────────────────
df_raw['amount_to_origin_ratio'] = df_raw['amount'] / (df_raw['origin_balance_before'] + 1e-9)

# ── Feature 4: Large Transfer Flag ───────────────────────────────
LARGE_THRESHOLD = 200_000
df_raw['is_large_transfer'] = (df_raw['amount'] > LARGE_THRESHOLD).astype(int)

# ── Feature 5: Transaction Type Encoded ─────────────────────────
TYPE_MAP = {'CASH_IN': 0, 'CASH_OUT': 1, 'DEBIT': 2, 'PAYMENT': 3, 'TRANSFER': 4}
df_raw['transaction_type_encoded'] = df_raw['transaction_type'].map(TYPE_MAP)

print('Features engineered. Summary:')
engineered = ['balance_diff_origin','balance_error_origin','balance_diff_dest',
              'amount_to_origin_ratio','is_large_transfer','transaction_type_encoded']
df_raw[engineered + ['is_fraud']].describe().round(3)

In [ ]:
# ── Feature Importance Visualization (Correlation with Target) ────
FEATURE_COLS = [
    'transaction_type_encoded','amount','origin_balance_before','origin_balance_after',
    'dest_balance_before','dest_balance_after','balance_diff_origin','balance_diff_dest',
    'amount_to_origin_ratio','is_large_transfer'
]

correlations = df_raw[FEATURE_COLS + ['is_fraud']].corr()['is_fraud'].drop('is_fraud').abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0.1 else '#3498db' for v in correlations.values]
plt.barh(correlations.index, correlations.values, color=colors, edgecolor='white')
plt.xlabel('|Correlation with Fraud Label|')
plt.title('Feature Correlation with Fraud Target', fontweight='bold', fontsize=14)
plt.axvline(0.1, color='gray', linestyle='--', alpha=0.7, label='0.1 threshold')
plt.legend()
plt.tight_layout()
plt.savefig(f'{PROC_PATH}feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Feature correlations:')
print(correlations.to_string())

In [ ]:
# ── Box Plots: Feature Distribution by Fraud Label ───────────────
plot_features = ['amount', 'amount_to_origin_ratio', 'balance_diff_origin', 'is_large_transfer']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feat in enumerate(plot_features):
    data_legit = df_raw[df_raw['is_fraud']==0][feat].clip(upper=df_raw[feat].quantile(0.99))
    data_fraud = df_raw[df_raw['is_fraud']==1][feat].clip(upper=df_raw[feat].quantile(0.99))
    axes[i].boxplot([data_legit, data_fraud], labels=['Legitimate','Fraudulent'],
                    patch_artist=True,
                    boxprops=dict(facecolor=['#2ecc71','#e74c3c'][0]),
                    medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(f'{feat}', fontweight='bold')
    axes[i].set_ylabel('Value (99th pct clipped)')
plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROC_PATH}feature_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save Enriched Processed Data ──────────────────────────────────
# This overwrites the previous preprocessed data with the enriched feature set
# (the 02_Preprocessing notebook already does the split and scaling;
# running notebook 03 adds balance_error_origin as an additional feature if desired)
print('Feature engineering complete.')
print('Final FEATURE_COLS for model training:', FEATURE_COLS)

# Update feature list JSON
with open(f'{PROC_PATH}feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f)
print(f'Updated {PROC_PATH}feature_columns.json')
print()
print('NOTE: Run notebook 02_Preprocessing.ipynb AGAIN after modifying features')
print('to regenerate scaled train/test arrays with the correct feature set.')